In [ ]:
pip install requests beautifulsoup4 spacy pandas newspaper3k

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 60.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 8.2 MB/s eta 0:00:00
  Created wheel for tinysegmenter: filename=tinysegmenter-0.3-py3-none-any.whl size=13540 sha256=42c7e1ef0dd44591d8024e87990dd88971da2c9bd389081028c119bea45e99be
  Stored in directory: /root/.cache/pip/wheels/fc/ab/f8/cce3a9ae6d828bd346be695f7ff54612cd22b7cbd7208d68f3
  Created wheel for feedfinder2: filename=feedfinder2-0.0.4-py3-none-any.whl size=3341 sha256=effed900754db465bde98bc052bff75f910e86c01cefc07c06278c160f80f726
  Stored in directory: /root/.cache/pip/wheels/80/d5/72/9cd9eccc819636436c6a6e59c22a0fb1ec

In [ ]:
!pip install -U spacy
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 65.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
!pip install newspaper3k
!pip install lxml_html_clean

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import re
import time
import random
import spacy
import pandas as pd
from datetime import datetime
from urllib.parse import urlparse
from newspaper import Article
from geopy.geocoders import Nominatim

In [ ]:
# Load SpaCy
nlp = spacy.load("en_core_web_sm")

class NewsArticleScraper:
    def __init__(self):
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
        }
        self.output_file = f"hyderabad_road_safety_articles_{datetime.now().strftime('%Y%m%d')}.csv"
        self.keywords = [
            "accident", "fatal", "injured", "crash", "collision",
            "death", "deceased", "killed", "road safety", "traffic", "vehicle", "pedestrian"
        ]
        self.news_sources = [
    "https://www.thehindu.com/news/cities/Hyderabad/",
    "https://telanganatoday.com/category/hyderabad",
    "https://www.deccanchronicle.com/nation/current-affairs/hyderabad",
    "https://timesofindia.indiatimes.com/city/hyderabad",
    "https://www.thenewsminute.com/telangana",
    "https://www.siasat.com/news/hyderabad/",
    "https://www.hindustantimes.com/cities/hyderabad-news",
    "https://www.indiaherald.com/Search/en/hyderabad"
        ]

        self.geolocator = Nominatim(user_agent="road_safety_scraper")

        # Create output CSV
        with open(self.output_file, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['Title', 'Date', 'Source', 'URL', 'Keywords Found', 'Location Details', 'Latitude', 'Longitude'])

    def extract_article_data(self, url):
        try:
            article = Article(url)
            article.download()
            article.parse()

            text = article.text
            found_keywords = [kw for kw in self.keywords if kw in text.lower()]
            if not found_keywords:
                return None

            doc = nlp(text)
            locations = list({ent.text for ent in doc.ents if ent.label_ in ["GPE", "LOC"] and ent.text.lower() not in ["hyderabad", "telangana", "india"]})

            lat, lon = None, None
            if locations:
                try:
                    geo = self.geolocator.geocode(f"{locations[0]}, Hyderabad, India", timeout=10)
                    if geo:
                        lat, lon = geo.latitude, geo.longitude
                except:
                    pass

            return {
                'title': article.title,
                'date': article.publish_date.strftime('%Y-%m-%d') if article.publish_date else "Unknown",
                'source': urlparse(url).netloc,
                'url': url,
                'keywords_found': ", ".join(found_keywords),
                'locations': ", ".join(locations),
                'lat': lat,
                'lon': lon
            }
        except Exception as e:
            print(f"Failed to process {url}: {e}")
            return None

    def check_news_sources(self):
        found_links = []
        for source in self.news_sources:
            try:
                for page in range(1, 31):
                    page_url = source if page == 1 else f"{source.rstrip('/')}/page-{page}/"
                    res = requests.get(page_url, headers=self.headers, timeout=10)
                    soup = BeautifulSoup(res.text, 'html.parser')
                    for a in soup.find_all('a', href=True):
                        href = a['href']
                        if not href.startswith('http'):
                            base = urlparse(source)
                            href = f"{base.scheme}://{base.netloc}{href}"

                        href_lower = href.lower()
                        anchor_text = a.text.strip().lower()

                        if (
                            "hyderabad" in href_lower and
                            any(kw in href_lower or kw in anchor_text for kw in self.keywords) and
                            not any(href_lower.endswith(path) for path in ["/hyderabad", "/city/hyderabad", "/tag/hyderabad"])
                        ):
                            found_links.append({'title': a.text.strip(), 'url': href})
            except Exception as e:
                print(f"Error scraping {source}: {e}")

        return list({v['url']: v for v in found_links}.values())

    def save_to_csv(self, data):
        with open(self.output_file, 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([
                data['title'],
                data['date'],
                data['source'],
                data['url'],
                data['keywords_found'],
                data['locations'],
                data['lat'],
                data['lon']
            ])

    def run(self):
        print("Scraping Hyderabad road safety articles...\n")
        articles = self.check_news_sources()
        print(f"Found {len(articles)} articles with potential relevance.\n")

        seen_urls = set()
        for i, article in enumerate(articles):
            if article['url'] in seen_urls:
                continue
            seen_urls.add(article['url'])

            print(f"📰 ({i+1}/{len(articles)}) Processing: {article['url']}")
            data = self.extract_article_data(article['url'])
            if data:
                self.save_to_csv(data)
            time.sleep(random.uniform(1, 3))

        print(f"\nFinished scraping. Data saved to: {self.output_file}")

if __name__ == "__main__":
    scraper = NewsArticleScraper()
    scraper.run()

Scraping Hyderabad road safety articles...

Found 6 articles with potential relevance.

📰 (1/6) Processing: https://www.thehindu.com/news/cities/Hyderabad/telangana-rolls-out-centralised-online-services-for-driving-licences-and-vehicle-registration/article69510730.ece
📰 (2/6) Processing: https://www.thehindu.com/news/cities/Hyderabad/speeding-goods-carrier-rams-into-nursing-students-in-gadwal-two-killed-two-injured/article69506343.ece
📰 (3/6) Processing: https://timesofindia.indiatimes.com/topic/hyderabad-traffic/news
📰 (4/6) Processing: https://timesofindia.indiatimes.com/city/hyderabad/7-killed-in-ap-temple-wall-collapse/articleshow/120775420.cms
📰 (5/6) Processing: https://timesofindia.indiatimes.com/city/hyderabad/man-bludgeoned-to-death-with-iron-rod-brother-in-law-held/articleshow/120774286.cms
📰 (6/6) Processing: https://timesofindia.indiatimes.com/city/hyderabad/hc-declines-to-order-release-of-seized-vehicle/articleshow/120739270.cms

Finished scraping. Data saved to: hyderabad